In [1]:
!pip -q install -U groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 6.1 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import warnings
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from groq import Groq

# =========================================================
# ENVIRONMENT SETUP
# =========================================================

warnings.filterwarnings("ignore")

LLM_MODEL = "llama-3.3-70b-versatile"

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def clean_secret(value: Optional[str]) -> str:
    if value is None:
        return ""
    return str(value).replace("\n", "").replace("\r", "").strip()

GROQ_API_KEY = None

if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    GROQ_API_KEY = input("Enter GROQ_API_KEY: ")

GROQ_API_KEY = clean_secret(GROQ_API_KEY)

if not GROQ_API_KEY.startswith("gsk_"):
    raise ValueError("Invalid Groq API key. It must start with 'gsk_'")

client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully.")
print(f"LLM model: {LLM_MODEL}")

# =========================================================
# SHARED STATE
# =========================================================

@dataclass
class ProposalState:
    rfp_text: str
    extracted_requirements: List[str] = field(default_factory=list)
    budget_constraints: List[str] = field(default_factory=list)
    evaluation_criteria: List[str] = field(default_factory=list)
    mapped_solutions: List[Dict[str, Any]] = field(default_factory=list)
    value_propositions: List[str] = field(default_factory=list)
    proposal_draft: str = ""
    qa_review: Dict[str, Any] = field(default_factory=dict)

# =========================================================
# COMPANY OFFERINGS CATALOG
# =========================================================

COMPANY_OFFERINGS = [
    {
        "name": "Cloud Analytics Suite",
        "keywords": ["analytics", "dashboard", "reporting", "business intelligence", "insights"],
        "description": "Enterprise-grade analytics platform with dashboards, reporting, and decision support.",
        "differentiator": "Fast deployment with prebuilt analytics templates and executive dashboards."
    },
    {
        "name": "AI Automation Platform",
        "keywords": ["automation", "ai", "workflow", "agent", "intelligent process automation"],
        "description": "AI-powered workflow automation system for enterprise process optimization.",
        "differentiator": "Reduces manual effort using AI agents and rule-based automation."
    },
    {
        "name": "CRM Integration Layer",
        "keywords": ["crm", "integration", "sales", "customer", "pipeline"],
        "description": "Secure integration framework connecting CRM, ERP, and internal systems.",
        "differentiator": "Seamless integration with minimal disruption to existing enterprise systems."
    },
    {
        "name": "Compliance and Security Module",
        "keywords": ["security", "compliance", "governance", "risk", "privacy"],
        "description": "Security and compliance controls with audit logging and policy enforcement.",
        "differentiator": "Built-in governance, access control, and compliance-ready reporting."
    }
]

# =========================================================
# LLM HELPERS
# =========================================================

def strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text)
    return text.strip()

def safe_json_loads(text: str) -> Dict[str, Any]:
    text = strip_code_fences(text)

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    raise ValueError(f"Model output is not valid JSON.\nRaw output:\n{text}")

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content or ""

# =========================================================
# BASE AGENT
# =========================================================

class BaseAgent:
    def __init__(self, name: str):
        self.name = name

    def run(self, state: ProposalState) -> ProposalState:
        raise NotImplementedError

# =========================================================
# AGENT 1: RFP ANALYZER
# =========================================================

class RFPAnalyzerAgent(BaseAgent):
    def __init__(self):
        super().__init__("RFP Analyzer")

    def run(self, state: ProposalState) -> ProposalState:
        system_prompt = """
You are an expert enterprise sales analyst.
Read the client's RFP and extract structured information.

Return ONLY valid JSON in this exact format:
{
  "extracted_requirements": ["..."],
  "budget_constraints": ["..."],
  "evaluation_criteria": ["..."]
}

Rules:
- Do not include markdown.
- Do not include explanations outside JSON.
- Keep requirements concise and specific.
- If a section is missing, return an empty list for that section.
""".strip()

        user_prompt = f"""
Analyze this RFP text and extract the key information.

RFP TEXT:
{state.rfp_text}
""".strip()

        raw_output = call_llm(system_prompt, user_prompt, temperature=0.1)
        parsed = safe_json_loads(raw_output)

        state.extracted_requirements = parsed.get("extracted_requirements", [])
        state.budget_constraints = parsed.get("budget_constraints", [])
        state.evaluation_criteria = parsed.get("evaluation_criteria", [])
        return state

# =========================================================
# AGENT 2: SOLUTION MAPPER
# =========================================================

class SolutionMapperAgent(BaseAgent):
    def __init__(self):
        super().__init__("Solution Mapper")

    def run(self, state: ProposalState) -> ProposalState:
        system_prompt = """
You are a solution architect for an enterprise sales team.
Your job is to map company offerings to client requirements.

Return ONLY valid JSON in this exact format:
{
  "mapped_solutions": [
    {
      "requirement": "...",
      "solution_name": "...",
      "description": "...",
      "differentiator": "...",
      "justification": "..."
    }
  ],
  "value_propositions": ["..."]
}

Rules:
- Use only the provided offerings catalog.
- Map each major requirement to the best matching offering.
- Keep justifications practical and business-focused.
- Do not include markdown.
- Do not include any text outside JSON.
""".strip()

        user_prompt = f"""
CLIENT REQUIREMENTS:
{json.dumps(state.extracted_requirements, indent=2)}

BUDGET CONSTRAINTS:
{json.dumps(state.budget_constraints, indent=2)}

EVALUATION CRITERIA:
{json.dumps(state.evaluation_criteria, indent=2)}

COMPANY OFFERINGS:
{json.dumps(COMPANY_OFFERINGS, indent=2)}

Generate the best solution mapping.
""".strip()

        raw_output = call_llm(system_prompt, user_prompt, temperature=0.2)
        parsed = safe_json_loads(raw_output)

        state.mapped_solutions = parsed.get("mapped_solutions", [])
        state.value_propositions = parsed.get("value_propositions", [])
        return state

# =========================================================
# AGENT 3: PROPOSAL WRITER
# =========================================================

class ProposalWriterAgent(BaseAgent):
    def __init__(self):
        super().__init__("Proposal Writer")

    def run(self, state: ProposalState) -> ProposalState:
        system_prompt = """
You are a senior enterprise proposal writer.
Write a polished, structured, client-specific sales proposal.

Rules:
- Use a professional business tone.
- Make the proposal clear, persuasive, and well organized.
- Include these sections:
  1. Executive Summary
  2. Client Requirements
  3. Proposed Solution
  4. Value Propositions
  5. Budget and Constraints
  6. Evaluation Alignment
  7. Conclusion
- Do not invent offerings that are not in the provided mapped solutions.
""".strip()

        user_prompt = f"""
Write the final proposal using the following structured inputs.

EXTRACTED REQUIREMENTS:
{json.dumps(state.extracted_requirements, indent=2)}

BUDGET CONSTRAINTS:
{json.dumps(state.budget_constraints, indent=2)}

EVALUATION CRITERIA:
{json.dumps(state.evaluation_criteria, indent=2)}

MAPPED SOLUTIONS:
{json.dumps(state.mapped_solutions, indent=2)}

VALUE PROPOSITIONS:
{json.dumps(state.value_propositions, indent=2)}
""".strip()

        raw_output = call_llm(system_prompt, user_prompt, temperature=0.3)
        state.proposal_draft = raw_output.strip()
        return state

# =========================================================
# AGENT 4: QA REVIEWER
# =========================================================

class QAReviewAgent(BaseAgent):
    def __init__(self):
        super().__init__("QA Reviewer")

    def run(self, state: ProposalState) -> ProposalState:
        system_prompt = """
You are a proposal quality reviewer.
Review the proposal for:
- consistency with requirements
- professionalism
- completeness
- clarity

Return ONLY valid JSON in this format:
{
  "qa_summary": "...",
  "missing_points": ["..."],
  "improvement_suggestions": ["..."]
}
""".strip()

        user_prompt = f"""
RFP TEXT:
{state.rfp_text}

PROPOSAL DRAFT:
{state.proposal_draft}
""".strip()

        raw_output = call_llm(system_prompt, user_prompt, temperature=0.1)
        parsed = safe_json_loads(raw_output)
        state.qa_review = parsed
        return state

# =========================================================
# WORKFLOW ORCHESTRATOR
# =========================================================

class SmartSalesProposalAssistant:
    def __init__(self, use_qa: bool = True):
        self.agents = [
            RFPAnalyzerAgent(),
            SolutionMapperAgent(),
            ProposalWriterAgent()
        ]
        if use_qa:
            self.agents.append(QAReviewAgent())

    def run(self, rfp_text: str) -> ProposalState:
        state = ProposalState(rfp_text=rfp_text)

        for agent in self.agents:
            print(f"Running: {agent.name}")
            state = agent.run(state)

        return state

# =========================================================
# USER INPUT INTERFACE
# =========================================================

print("\nSMART SALES PROPOSAL ASSISTANT")
print("=" * 70)
print("Paste the client RFP text below.")
print("The system will analyze it, map solutions, generate a proposal, and run QA.\n")

rfp_text = input("Enter / Paste the client RFP text:\n\n")

if not rfp_text.strip():
    print("\nError: RFP text cannot be empty.")
else:
    print("\nProcessing request...\n")

    assistant = SmartSalesProposalAssistant(use_qa=True)
    result = assistant.run(rfp_text)

    print("\n" + "=" * 80)
    print("EXTRACTED REQUIREMENTS")
    print("=" * 80)
    print(json.dumps(result.extracted_requirements, indent=2))

    print("\n" + "=" * 80)
    print("BUDGET CONSTRAINTS")
    print("=" * 80)
    print(json.dumps(result.budget_constraints, indent=2))

    print("\n" + "=" * 80)
    print("EVALUATION CRITERIA")
    print("=" * 80)
    print(json.dumps(result.evaluation_criteria, indent=2))

    print("\n" + "=" * 80)
    print("MAPPED SOLUTIONS")
    print("=" * 80)
    print(json.dumps(result.mapped_solutions, indent=2))

    print("\n" + "=" * 80)
    print("VALUE PROPOSITIONS")
    print("=" * 80)
    print(json.dumps(result.value_propositions, indent=2))

    print("\n" + "=" * 80)
    print("FINAL GENERATED PROPOSAL")
    print("=" * 80)
    print(result.proposal_draft)

    print("\n" + "=" * 80)
    print("QUALITY REVIEW")
    print("=" * 80)
    print(json.dumps(result.qa_review, indent=2))

Groq client initialized successfully.
LLM model: llama-3.3-70b-versatile

SMART SALES PROPOSAL ASSISTANT
Paste the client RFP text below.
The system will analyze it, map solutions, generate a proposal, and run QA.

Enter / Paste the client RFP text:

You are an enterprise sales assistant. Analyze the following RFP (Request for Proposal) and perform the following tasks:  1. Extract key client requirements, budget constraints, and evaluation criteria. 2. Map the most suitable company solutions to each requirement. 3. Generate a professional, well-structured sales proposal with clear sections:    - Executive Summary    - Client Requirements    - Proposed Solution    - Value Propositions    - Budget and Constraints    - Evaluation Alignment    - Conclusion 4. Ensure the proposal is tailored, persuasive, and aligned with enterprise standards. 5. Do not make assumptions beyond the provided RFP.  RFP:

Processing request...

Running: RFP Analyzer
Running: Solution Mapper
Running: Proposal Wri